# Results analysis — P2P-Thief (anrbj666)

Sensitivity analysis (OAT) over the parameters that shape the pursuit game.
Raw data: `results/experiments/sensitivity.json`; regenerate everything with
`uv run python scripts/run_sensitivity.py`.

## 1. Scent decay — why $\rho = 0.10$ is the game's memory

The trail obeys $\tau_t = \tau_0 (1-\rho)^t$ with $\tau_0 = 0.9$. The
lie-detection floor (0.4) is crossed after
$t^* = \frac{\ln(0.4/0.9)}{\ln(1-\rho)}$ turns — at the fixed $\rho=0.10$
that is $t^* \approx 7.7$: a trail stays *legally readable* for ~7 turns,
long enough to catch lies one full round later, short enough that history
does not drown the present (the book's local-optima argument, ch. 4).

![decay](../assets/sens_decay.png)

In [1]:
import json
import math
from pathlib import Path

sens = json.loads(Path("../results/experiments/sensitivity.json").read_text())
for rho in (0.05, 0.10, 0.20, 0.30):
    t_star = math.log(0.4 / 0.9) / math.log(1 - rho)
    print(f"rho={rho}: readable for ~{t_star:.1f} turns")

rho=0.05: readable for ~15.8 turns
rho=0.1: readable for ~7.7 turns
rho=0.2: readable for ~3.6 turns
rho=0.3: readable for ~2.3 turns


## 2. Hint honesty — deception does not save a weak runner

Blind pursuit (belief-only) vs a random thief across honesty levels
$P(\text{truth}) \in \{0, .25, .5, .75, 1\}$, 15 seeds each. Capture rate
stays in the 0.93–1.0 band **independent of honesty**: the scent evidence
dominates the posterior, so lying only helps a thief whose *movement* also
exploits the belief. This validated our decision to invest in movement
strategy over prompt sophistication (moves are pure Python anyway — rule 25).

![honesty](../assets/sens_honesty.png)

In [2]:
print(json.dumps(sens["honesty_capture_rate"], indent=2))

{
  "0.0": 1.0,
  "0.25": 0.8666666666666667,
  "0.5": 0.8666666666666667,
  "0.75": 1.0,
  "1.0": 1.0
}


## 3. Board size — every added cell is thief territory

Mean turns-to-capture under full information grows superlinearly-ish with
the board side (7.7 → 9.3 → 12.1 for 7/9/11): the state space of the
Dec-POMDP grows as $O(n^4)$ (two positions) times barrier configurations,
while the survival threshold stays 35 — so negotiating a larger board is a
pro-thief move, and our cop should resist raising the minimum.

![board](../assets/sens_board.png)

In [3]:
print(sens["board_capture_turns"])

{'7': 7.5, '9': 9.3, '11': 12.1}


## Conclusions

1. Keep $ho=0.10$ (fixed anyway) — the ~7-turn readable window is what
   makes the (1−ρ)·0.9 lie test decisive.
2. Strategy budget goes to movement, not rhetoric: honesty sweeps show the
   verbal layer cannot rescue weak evasion against a belief-driven pursuer.
3. Board-size negotiation is strategic: cop wants 7×7, thief wants bigger.

References: Bernstein et al. (Dec-POMDP complexity); Theraulaz & Bonabeau
(stigmergy); the course rulebook ch. 4–6.

## 4. The RL campaign — an adversarial arms race with promotion gates

Full narrative in the README; every number below is loaded from the committed
experiment artifacts (`results/experiments/`). The campaign's shape: linear
Q-learning → Double-DQN with trap-threat features → two-round arms race via
weight-data crossover between the twin repos → hyperparameter sweep → two
gated promotions, both correctly rejected. Negative results are first-class
artifacts here: they carry the campaign's strongest claims.


In [4]:
linear = json.loads(Path('../results/experiments/rl_training.json').read_text(encoding='utf-8'))
deep = json.loads(Path('../results/experiments/deep_rl_training.json').read_text(encoding='utf-8'))
ens = json.loads(Path('../results/experiments/deep_rl_training_v2_ensemble.json').read_text(encoding='utf-8'))
fine = json.loads(Path('../results/experiments/deep_rl_finetune.json').read_text(encoding='utf-8'))
print('linear: from-scratch vs informed prior curves recorded:', list(linear['runs']))
print('deep v1 vs learned trap cop :', deep['final_survival_vs_deep_cop'])
print('v2 ensemble retrain         :', ens['final_100_game_evals']['vs_learned_trap_cop'], '(collapse, recorded)')
print('fine-tune verdict           :', fine['shipped'])
print('knife-edge                  :', fine['v1_baseline'])


linear: from-scratch vs informed prior curves recorded: ['from_scratch', 'informed_prior']
deep v1 vs learned trap cop : {'win_rate': 1.0, 'games': 100}
v2 ensemble retrain         : 0.06 (collapse, recorded)
fine-tune verdict           : v1 unchanged (best checkpoint was episode 0 - the untouched v1)
knife-edge                  : {'vs_learned_trap_cop': 1.0, 'vs_learned_trap_cop_noisy': 0.0, 'vs_heuristic_trapcop': 1.0}
